In [1]:
import re
from typing import Tuple, List

_OP_SYM = {"ADD": "+", "SUB": "-", "MUL": "x", "DIV": "÷"}  # raw symbol map

class OptParseError(ValueError):
    pass

def _tokenize(s: str) -> List[str]:
    # simple whitespace tokenization (your OPT is space-delimited)
    return [t for t in s.strip().split() if t]

def _parse_expr(tokens: List[str], i: int) -> Tuple[str, int]:
    """
    Recursive descent for your prefix operator form:
      expr := bin | atom
      bin  := (ADD|SUB|MUL|DIV) expr expr
      atom := FRAC int int
            | INT int
            | MIX int FRAC int int
            | -MIX int FRAC int int
            | UNDEF
    Returns: (raw_formula_string, next_index)
    """
    if i >= len(tokens):
        raise OptParseError("Unexpected end of tokens.")

    tok = tokens[i]

    # Binary op (prefix)
    if tok in ("ADD", "SUB", "MUL", "DIV"):
        left, j = _parse_expr(tokens, i + 1)
        right, k = _parse_expr(tokens, j)
        sym = _OP_SYM[tok]
        return f"{left} {sym} {right}", k

    # Atoms
    if tok == "FRAC":
        if i + 2 >= len(tokens):
            raise OptParseError("FRAC expects two integers.")
        n, d = tokens[i + 1], tokens[i + 2]
        # allow signed numerator; denominator must be positive integer in your gen
        if not re.fullmatch(r"[+-]?\d+", n) or not re.fullmatch(r"\d+", d):
            raise OptParseError(f"Bad FRAC arguments: {n} {d}")
        return f"{int(n)}/{int(d)}", i + 3

    if tok == "INT":
        if i + 1 >= len(tokens) or not re.fullmatch(r"[+-]?\d+", tokens[i + 1]):
            raise OptParseError("INT expects a signed integer.")
        return str(int(tokens[i + 1])), i + 2

    if tok in ("MIX", "-MIX"):
        neg = (tok == "-MIX")
        # Expect: (sign)MIX whole FRAC num den
        if i + 4 >= len(tokens) or tokens[i + 2] != "FRAC":
            raise OptParseError("MIX expects: MIX <whole> FRAC <num> <den>.")
        whole = tokens[i + 1]
        num   = tokens[i + 3]
        den   = tokens[i + 4]
        if not re.fullmatch(r"\d+", whole):
            raise OptParseError(f"Bad MIX whole: {whole}")
        if not (re.fullmatch(r"[+-]?\d+", num) and re.fullmatch(r"\d+", den)):
            raise OptParseError(f"Bad MIX fraction: {num}/{den}")
        sgn = "-" if neg else ""
        return f"{sgn}{int(whole)} {abs(int(num))}/{int(den)}", i + 5

    if tok == "UNDEF":
        return "Undefined", i + 1

    # Fallback: if someone serializes "ANS something" (shouldn't happen here)
    if tok == "ANS":
        # swallow the rest as one display token
        return " ".join(tokens[i:]), len(tokens)

    raise OptParseError(f"Unexpected token: {tok}")

def opt_to_formula(opt: str) -> str:
    """
    Convert an OPT string into a raw formula like:
      '4/17 - 3/7 = 7/24'  or  '2 1/3 x 5/7'
    Recognizes either:
      LHS_tokens EQ RHS_tokens      (infix equality)
      <expression>                  (single expression; no '=')
    """
    tokens = _tokenize(opt)
    if not tokens:
        raise OptParseError("Empty OPT string.")

    # Split on a single infix EQ (per your dataset)
    if "EQ" in tokens:
        eq_idx = tokens.index("EQ")
        lhs_tokens = tokens[:eq_idx]
        rhs_tokens = tokens[eq_idx + 1:]
        if not lhs_tokens or not rhs_tokens:
            raise OptParseError("Malformed equation around EQ.")
        lhs, i = _parse_expr(lhs_tokens, 0)
        if i != len(lhs_tokens):
            raise OptParseError("Extra tokens on LHS after parsing.")
        rhs, j = _parse_expr(rhs_tokens, 0)
        if j != len(rhs_tokens):
            raise OptParseError("Extra tokens on RHS after parsing.")
        return f"{lhs} = {rhs}"

    # No equality: parse single expression
    expr, i = _parse_expr(tokens, 0)
    if i != len(tokens):
        raise OptParseError("Extra tokens after parsing expression.")
    return expr

In [4]:
assert opt_to_formula("SUB FRAC 4 17 FRAC 3 7 EQ FRAC 7 24") == "4/17 - 3/7 = 7/24"
assert opt_to_formula("ADD MIX 2 FRAC 1 3 FRAC 5 6") == "2 1/3 + 5/6"
assert opt_to_formula("MUL INT -7 FRAC 3 5") == "-7 x 3/5"
assert opt_to_formula("DIV FRAC -3 7 FRAC 5 2") == "-3/7 ÷ 5/2"
assert opt_to_formula("-MIX 1 FRAC 2 5") == "-1 2/5"
assert opt_to_formula("FRAC -7 24") == "-7/24"
assert opt_to_formula("UNDEF") == "Undefined"